# Suite TRN — Translators

Every pair of dialects has its own bidirectional translator (`@mbse/expressions/Translators`), declared as rules
that pair two patterns with holes. Translating co-traverses each rule's pattern with the expression to bind the
holes, then instantiates the other side, so every node is translated once and sharing is kept. A dialect with no
binding gets a let's value substituted for its name, and imports are dropped where the other side cannot declare
them, or added where the translation needs them (`import numpy as np`). Translations keep an expression's form;
its value follows each dialect's rules, and this suite shows where those agree and where they do not. (Evaluating
the NumPy style with numpy is the Python implementation's only.)

In [ ]:
import { Evaluators as V, Expressions as E } from "@mbse/expressions";
import * as Excel from "@mbse/expressions/Dialects/Excel";
import * as Latex from "@mbse/expressions/Dialects/Latex";
import * as Matlab from "@mbse/expressions/Dialects/Matlab";
import * as Python from "@mbse/expressions/Dialects/Python";
import { Terms as F } from "@mbse/expressions/Framework";
import { Elide, Hole, Inline, Pairwise, Pattern, Prelude, Rule, holes, renames } from "@mbse/expressions/Framework/Translators";
import * as Translators from "@mbse/expressions/Translators";
import { Errors, Proxies, Schemas as S } from "@mbse/schemas/Framework";

const { AttributeError, KeyError, LookupError, ValueError } = Errors;
import { assert, raises, same, text } from "./support.js";

const [P, M, X, L] = [Python.Expressions, Matlab.Expressions, Excel.Expressions, Latex.Expressions];
const [PV, MV, XV] = [Python.Evaluators, Matlab.Evaluators, Excel.Evaluators];
const [BASIC, PYTHON, MATLAB, EXCEL, LATEX] = [E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT];
const DIALECTS = [BASIC, PYTHON, MATLAB, EXCEL, LATEX];
const EVALUATE = new Map<F.Declared, (e: unknown, scope?: any) => unknown>([
  [BASIC, V.OfAny as any], [PYTHON, PV.OfAny], [MATLAB, MV.OfAny], [EXCEL, XV.OfAny]]);
const NUMPY = Translators.Basic_Python.NUMPY;
const self = E.variable("this");
const age = self.age;
const core = age.ge(18n).and_(age.lt(65n).or_(self.has("email").not_())).and_(self.name.ne("x")).or_(
  age.add(1n).sub(2n).mul(3n).neg().gt(0n).and_(age.eq(1n).not_())).and_(self.has("name").eq(true)).and_(
  E.literal(1.5).le(2.5)); // every translatable operation and literal type, operands of one type, no let
const rule = E.let_("a", age, E.variable("a").ge(18n).implies(self.has("email"))); // a let and implies
const permutations = (items: F.Declared[], n: number): F.Declared[][] =>
  n === 0 ? [[]] : items.flatMap((item) => permutations(items.filter((i) => i !== item), n - 1).map((rest) => [item, ...rest]));
const kindOf = (dialect: F.Declared, tag: string) => dialect.kinds().get(tag) as any;
const json = (value: unknown) => globalThis.JSON.stringify(value);
/** The translation of `core` (or another Basic expression) into `dialect`. */
const into = (dialect: F.Declared, expression: any = core) =>
  (dialect === BASIC ? E.OfAny.resolve(expression) : Translators.between(BASIC, dialect).forward(expression));

const Person = new S.OfObject.Builder().ref().properties(text("name"), text("age", BigInt), text("email")).create();
Proxies.register("Person", Person);
const B = (Proxies.Builders as any).Person;
const people = [B().name("Ann").age(30n).email("a@x").create(), B().name("Bob").age(70n).create(),
  B().name("x").age(10n).email("c@x").create()];

## TRN-01 · Rules pair patterns; holes bind arguments and attributes

In [ ]:
{
  const [A, B_] = holes("A", "B") as [Hole, Hole];
  const K = new Hole("K", String);
  const double = new Pattern("operation", { name: "add" }, A, A);
  assert(double.size() === 2 && String(double) === "operation(A, A, name='add')" && json([...double.holes()]) === json([["A", "argument"]]));
  const has = new Pattern("operation", { name: "has" }, A, new Pattern("literal", { value: K }));
  assert(has.size() === 3 && json([...has.holes()]) === json([["A", "argument"], ["K", "attribute"]]));
  raises(ValueError, () => new Rule(new Pattern("operation", { name: "f" }, A), new Pattern("operation", { name: "f" }, A, B_)),
    "operation(A, B, name='f') needs the argument hole B, which operation(A, name='f') does not bind");
  raises(ValueError, () => new Rule(new Pattern("operation", { name: "f" }, A), new Pattern("variable", { name: A })),
    "variable(name=A) needs the attribute hole A");
  new Rule(new Pattern("operation", { name: "f" }, A), new Pattern("operation", { name: "f" }, A, B_), "backward"); // only the other way needs B
  raises(ValueError, () => new Rule(double, double, "up"), "direction must be 'both', 'forward' or 'backward', got 'up'");
  for (const declare of [() => new Inline("let", "up"), () => new Elide("import", "up"), () => new Prelude(new Pattern("import", {}, A), "up")]) {
    raises(ValueError, declare, "side must be 'left' or 'right', got 'up'");
  }
  for (const pattern of [new Pattern("import", { module: "numpy" }), new Pattern("import", {}, new Pattern("name", { name: "x" })),
    new Pattern("import", { module: new Hole("M") }, A)]) {
    raises(ValueError, () => new Prelude(pattern, "right"), "a prelude is an import whose one argument is a hole");
  }
  assert(json(renames("operation", "name", "call", "function", { eq: "equal" }, 2).map((r) =>
    [Object.fromEntries(r.left.attributes), Object.fromEntries(r.right.attributes)])) === json([[{ name: "eq" }, { function: "equal" }]]));

  // A translator within one dialect is a rewriting: holes bound twice must bind the same node, or equal values.
  const simplify = new Pairwise(BASIC, BASIC, [
    new Rule(double, new Pattern("operation", { name: "mul" }, A, new Pattern("literal", { value: 2n })), "forward"),
    new Rule(new Pattern("operation", { name: "eq" }, new Pattern("literal", { value: K }), new Pattern("literal", { value: K })),
      new Pattern("literal", { value: true }), "forward"),
    new Rule(new Pattern("operation", { name: "add" }, A, B_), new Pattern("operation", { name: "add" }, A, B_), "forward"),
    new Rule(new Pattern("operation", { name: "eq" }, new Pattern("literal", { value: new Hole("V") }), new Pattern("literal", { value: new Hole("W") })),
      new Pattern("literal", { value: false }), "forward"),
    new Rule(new Pattern("variable", { name: new Hole("N") }), new Pattern("variable", { name: new Hole("N") })),
    new Rule(new Pattern("literal", { value: new Hole("V") }), new Pattern("literal", { value: new Hole("V") }), "forward"),
  ]);
  const [x, y] = [E.variable("x"), E.variable("y")];
  assert(F.same(simplify.forward(x.add(x)), x.mul(2n).data) && F.same(simplify.forward(x.add(y)), x.add(y).data));
  assert(simplify.forward(E.literal("a").eq("a")).value === true && simplify.forward(E.literal("a").eq("b")).value === false);
  assert(simplify.forward(E.literal(1n).eq(1n)).value === false); // K holds only a str
  assert(String(simplify) === "<translator Basic <-> Basic>");
}

## TRN-02 · Every pair translates both ways; forms round-trip where both dialects have them

In [ ]:
{
  assert(Translators.TRANSLATORS.length === 10);
  for (const [source, target] of permutations(DIALECTS, 2) as [F.Declared, F.Declared][]) {
    const translator = Translators.between(source, target);
    assert(translator.left() === source && translator.right() === target);
    const start = into(source);
    const there = translator.forward(start);
    assert(there.dialect() === target && there.validate({ bound: ["this"], core: true }).length === 0, `${source.name()} ${target.name()}`);
    assert(F.same(translator.backward(there), start), `${source.name()} ${target.name()}`); // round trip
    assert(F.same(translator.inverse().backward(start), there));
  }
  raises(LookupError, () => Translators.between(BASIC, BASIC), "no translator between Basic and Basic");

  assert(P.render(Translators.between(BASIC, PYTHON).forward(rule)) === "(lambda a: hasattr(this, 'email') if a >= 18 else True)(this.age)");
  assert(M.render(Translators.between(BASIC, MATLAB).forward(rule)) === '~(this.age >= 18) || isfield(this, "email")');
  assert(X.render(Translators.between(BASIC, EXCEL).forward(rule)) === "=LET(a, this.age, IF(a >= 18, NOT(ISERROR(this.email)), TRUE))");
  assert(L.render(Translators.between(BASIC, LATEX).forward(rule))
    === "a \\geq 18 \\implies \\operatorname{has}(\\mathit{this}, \\text{email}) \\quad \\text{where } a = \\mathit{this}.\\mathit{age}");
  assert(L.render(Translators.between(PYTHON, LATEX).forward(P.binop("/", P.name("x"), 2n))) === "\\frac{x}{2}");
  // Python, Excel and Latex have lets and implication, so the rule round-trips; Matlab inlines the let and writes
  // implies with ||.
  for (const other of [PYTHON, EXCEL, LATEX]) {
    assert(F.same(Translators.between(other, BASIC).forward(Translators.between(BASIC, other).forward(rule)), rule.data));
  }
  const back = Translators.between(MATLAB, BASIC).forward(Translators.between(BASIC, MATLAB).forward(rule));
  assert(F.same(back, age.ge(18n).not_().or_(self.has("email")).data));

  // NumPy style is a second translator between Basic and Python: numpy's functions, over columns, importing numpy.
  const styled = NUMPY.forward(rule);
  assert(P.render(styled) === "import numpy as np\n(lambda a: np.where(np.greater_equal(a, 18), "
    + "np.logical_not(np.ma.getmaskarray(this['email'])), True))(this['age'])");
  assert(F.same(NUMPY.backward(styled), rule.data) && F.same(NUMPY.backward(NUMPY.forward(core)), core.data));
  assert(P.render(NUMPY.forward(E.literal(1n).add(2n))) === "import numpy as np\nnp.add(1, 2)");
  assert(P.render(NUMPY.forward(E.literal(1n))) === "1" && P.render(NUMPY.forward(self)) === "this"); // no np, no import
  assert(F.same(NUMPY.inverse().forward(styled), rule.data) && NUMPY.inverse().backward(E.literal(1n)).value === 1n);
  const imported = P.import_("math", P.importfrom("numpy", "pi", P.compare(">=", P.attribute(P.name("this"), "age"), 18n)));
  // imports the other side cannot declare are dropped
  for (const target of [BASIC, MATLAB, EXCEL]) {
    assert(F.same(Translators.between(PYTHON, target).forward(imported), into(target, age.ge(18n))));
  }
  assert(X.render(Translators.between(MATLAB, EXCEL).forward(M.import_("pkg.*", M.binary(">", 1n, 0n)))) === "=1 > 0");
  assert(F.same(Translators.between(MATLAB, BASIC).forward(M.import_("pkg.*", M.constant(1n))), E.literal(1n).data));
  raises(ValueError, () => Translators.between(PYTHON, MATLAB).forward(new (kindOf(PYTHON, "import"))("math")), "an import needs a body");
  raises(ValueError, () => Translators.between(PYTHON, BASIC).forward(P.import_("numpy", P.call("np.add", 1n, 2n), "np")),
    "Python call has no Basic counterpart"); // read by NUMPY
}

## TRN-03 · Translating the same pair of dialects by any route gives the same expression

In [ ]:
for (const [a, b, c] of permutations(DIALECTS, 3) as [F.Declared, F.Declared, F.Declared][]) {
  const start = into(a);
  const direct = Translators.between(a, c).forward(start);
  const routed = Translators.between(b, c).forward(Translators.between(a, b).forward(start));
  assert(F.same(direct, routed), `${a.name()} ${b.name()} ${c.name()}`);
}

## TRN-04 · Co-traversal translates each node once, keeps sharing, and traces the correspondence

In [ ]:
{
  const shared = E.operation("add", age, age).data;
  let trace: [any, any][] = [];
  const out = Translators.between(BASIC, EXCEL).forward(shared, trace);
  assert(out.left === out.right); // the shared argument is translated once
  assert(json(trace.map(([s, t]) => [s.kind().KIND, t.kind().KIND])) === json([["variable", "name"], ["operation", "field"],
    ["operation", "infix"]]));
  assert(trace[2]?.[0] === shared && trace[2]?.[1] === out && trace[1]?.[1] === out.left);
  const a = E.variable("a");
  trace = [];
  const inlined = Translators.between(BASIC, MATLAB).forward(E.let_("a", age, a.add(a)), trace);
  assert(M.render(inlined) === "this.age + this.age" && inlined.left === inlined.right);
  assert(same(trace.map(([s]) => s.kind().KIND), ["variable", "operation", "variable", "operation", "let"]));
  const shadowed = E.let_("a", 1n, E.variable("a").add(E.let_("a", 2n, E.variable("a")))); // the innermost binding wins
  assert(M.render(Translators.between(BASIC, MATLAB).forward(shadowed)) === "1 + 2");
  assert(MV.OfAny(Translators.between(BASIC, MATLAB).forward(shadowed)) === 3 && V.OfAny(shadowed) === 3n);
  assert(M.render(Translators.between(EXCEL, MATLAB).forward(X.let_("n", 2n, X.infix("*", X.name("n"), X.name("n"))))) === "2 .* 2");
  const loop = new E.OfOperation.Builder().name("not").create();
  new E.OfOperation.Builder(loop).arguments(loop).update();
  raises(ValueError, () => Translators.between(BASIC, MATLAB).forward(loop), "the expression contains a cycle");
  raises(TypeError, () => Translators.between(BASIC, MATLAB).forward(new E.OfOperation.Data("not", [7n])), "not an expression: 7");
  raises(ValueError, () => Translators.between(BASIC, MATLAB).forward(new E.OfLet.Data("a")), "a let needs a value");
  raises(ValueError, () => Translators.between(EXCEL, MATLAB).forward(new (kindOf(EXCEL, "let"))("a", X.constant(1n))), "a let needs a body");
}

## TRN-05 · What a dialect cannot say is refused, naming the node

In [ ]:
for (const [translate, expression, message] of [
  [(e: any) => Translators.between(BASIC, EXCEL).forward(e), E.literal(new Uint8Array([0])), "Basic literal b'\\x00' has no Excel counterpart"],
  [(e: any) => Translators.between(BASIC, PYTHON).forward(e), new E.OfLiteral.Data(), "Basic literal None has no Python counterpart"],
  [(e: any) => Translators.between(BASIC, MATLAB).forward(e), E.operation("in", 1n, 2n), "Basic operation 'in' has no Matlab counterpart"],
  [(e: any) => Translators.between(BASIC, PYTHON).forward(e), self.get(E.variable("key")), "Basic operation 'get' has no Python counterpart"],
  [(e: any) => NUMPY.forward(e), new E.OfOperation.Data("has", [self.data, 7n]), "Basic operation 'has'"],
  [(e: any) => Translators.between(BASIC, PYTHON).forward(e), E.operation("eq", 1n, 2n, 3n), "Basic operation 'eq' has no Python counterpart"],
  [(e: any) => Translators.between(EXCEL, BASIC).forward(e), X.function("ISERROR", 1n), "Excel function 'ISERROR' has no Basic counterpart"],
  [(e: any) => Translators.between(EXCEL, BASIC).forward(e), X.function("IF", true, 1n, 2n), "Excel function 'IF' has no Basic counterpart"],
  [(e: any) => Translators.between(EXCEL, BASIC).forward(e), new (kindOf(EXCEL, "let"))("a", X.constant(1n)), "Excel let has no Basic counterpart"],
  [(e: any) => Translators.between(PYTHON, MATLAB).forward(e), P.ifexp(true, 1n, 2n), "Python ifexp has no Matlab counterpart"],
  [(e: any) => Translators.between(EXCEL, MATLAB).forward(e), X.cell("A1"), "Excel cell has no Matlab counterpart"],
  [(e: any) => Translators.between(BASIC, LATEX).forward(e), E.literal(new Uint8Array([0])), "Basic literal b'\\x00' has no Latex counterpart"],
  [(e: any) => Translators.between(LATEX, BASIC).forward(e), L.frac(1n, 2n), "Latex frac has no Basic counterpart"],
  [(e: any) => Translators.between(PYTHON, LATEX).forward(e), P.ifexp(true, 1n, 2n), "Python ifexp has no Latex counterpart"],
  [(e: any) => Translators.between(MATLAB, BASIC).forward(e), M.call("isfield", M.identifier("s"), M.identifier("k")), "Matlab call 'isfield'"],
] as [(e: unknown) => unknown, unknown, string][]) {
  raises(ValueError, () => translate(expression), message);
}

## TRN-06 · Translations evaluate alike where the dialects' rules agree, and apart where they do not

In [ ]:
{
  const truthy = (value: unknown) => PV.truthy(value);
  const EVALUATED = DIALECTS.filter((d) => EVALUATE.has(d)); // Latex has no evaluator
  for (const person of people) {
    for (const expression of [core, rule]) {
      const values = EVALUATED.map((d) => truthy(EVALUATE.get(d)?.(into(d, expression), { this: person })));
      assert(new Set(values).size === 1, `${person.name} ${values}`);
    }
  }
  // NumPy evaluates all three at once, over columns: the Python implementation's only.

  const nobody = B().name("Nobody").create(); // no age: each dialect's own notion of a missing value
  const missing = age.ge(18n);
  assert(V.OfAny(missing, { this: nobody }) === null); // Basic: unknown
  raises(AttributeError, () => PV.OfAny(into(PYTHON, missing), { this: nobody }), "'Person' object has no attribute 'age'"); // Python raises
  assert(XV.OfAny(into(EXCEL, missing), { this: nobody }) === XV.Error.of("#FIELD!")); // an error value
  raises(KeyError, () => MV.OfAny(into(MATLAB, missing), { this: nobody }), 'Unrecognized field name "age".'); // MATLAB raises
  const [mixed, cased] = [E.literal(1n).eq(1.0), E.literal("a").eq("A")];
  assert(V.OfAny(mixed) === null && V.OfAny(cased) === false); // Basic never coerces, and compares text exactly
  assert(same(EVALUATED.slice(1).map((d) => truthy(EVALUATE.get(d)?.(into(d, mixed)))), [true, true, true]));
  assert(same(EVALUATED.slice(1).map((d) => truthy(EVALUATE.get(d)?.(into(d, cased)))), [false, false, true]));
}

## TRN-07 · The bitwise operations translate between Basic, Python, MATLAB and Excel, and evaluate alike on non-negative integers

In [ ]:
{
  const x = E.variable("x");
  const bits = x.bitand(12n).bitor(x.shr(2n)).bitxor(x.shl(1n));
  const RENDER = new Map<F.Declared, (e: any) => string>([[PYTHON, P.render], [MATLAB, M.render], [EXCEL, X.render]]);
  const expected = new Map<F.Declared, string>([[PYTHON, "(x & 12 | x >> 2) ^ x << 1"],
    [MATLAB, "bitxor(bitor(bitand(x, 12), bitshift(x, -2)), bitshift(x, 1))"],
    [EXCEL, "=BITXOR(BITOR(BITAND(x, 12), BITRSHIFT(x, 2)), BITLSHIFT(x, 1))"]]);
  for (const [dialect, source] of expected) {
    const translator = Translators.between(BASIC, dialect);
    const translated = translator.forward(bits);
    assert(RENDER.get(dialect)!(translated) === source && F.same(translator.inverse().forward(translated), bits.data));
    assert(Number(EVALUATE.get(dialect)!(translated, { x: 13n })) === 21 && V.OfAny(bits, { x: 13n }) === 21n);
  }
  // Between the other dialects too, by any route.
  for (const [a, b] of [[PYTHON, MATLAB], [PYTHON, EXCEL], [MATLAB, EXCEL]] as [F.Declared, F.Declared][]) {
    const source = Translators.between(BASIC, a).forward(bits);
    const target = Translators.between(a, b).forward(source);
    assert(F.same(target, Translators.between(BASIC, b).forward(bits)) && F.same(Translators.between(b, a).forward(target), source));
  }
  // bitnot is Python's ~ and NumPy's invert; MATLAB and Excel have none for their numbers.
  assert(P.render(Translators.between(BASIC, PYTHON).forward(x.bitnot())) === "~x");
  assert(P.render(NUMPY.forward(x.bitnot().shl(1n))) === "import numpy as np\nnp.left_shift(np.invert(x), 1)");
  raises(ValueError, () => Translators.between(BASIC, MATLAB).forward(x.bitnot()), "Basic operation 'bitnot' has no Matlab counterpart");
  raises(ValueError, () => Translators.between(BASIC, EXCEL).forward(x.bitnot()), "Basic operation 'bitnot' has no Excel counterpart");
  raises(ValueError, () => Translators.between(PYTHON, MATLAB).forward(P.unaryop("~", P.name("x"))),
    "Python unaryop '~' has no Matlab counterpart");
  // Basic's and Python's integers are unbounded and signed; MATLAB's and Excel's bit functions are not.
  const mask = x.bitand(1n);
  assert(V.OfAny(mask, { x: -1n }) === 1n && PV.OfAny(Translators.between(BASIC, PYTHON).forward(mask), { x: -1n }) === 1n);
  assert(XV.OfAny(Translators.between(BASIC, EXCEL).forward(mask), { x: -1n }) === XV.Error.of("#NUM!"));
}